# Phase 2 - Beyond SOTA Segment Discovery for Classification

Notebook nuevo orientado a encontrar trozos/secciones importantes para luego clasificar.

Pipeline incluido:
- Multi-metodo de descubrimiento de segmentos: pivots/impulsos, compresion-breakout, CUSUM shocks.
- Matrix Profile opcional (si `stumpy` esta instalado).
- Scoring unificado + deduplicacion por solapamiento (NMS temporal).
- Features de segmento + embedding de trayectoria (secuencia normalizada) listo para ML.
- Pseudo-labeling y export a `parquet` + `npz`.


## Nota de enfoque

"SOTA/Beyond SOTA" real depende de benchmark, leakage control y validacion out-of-sample.
Este notebook maximiza practicidad y robustez en tu entorno actual, sin depender de librerias faltantes.


In [ ]:
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from IPython.display import display

try:
    import stumpy
    HAS_STUMPY = True
except Exception:
    HAS_STUMPY = False

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)
warnings.filterwarnings('ignore')

print('HAS_STUMPY =', HAS_STUMPY)


In [ ]:
# =========================
# Config
# =========================
DATA_DIR = Path('./data/parquet_h1')
SYMBOLS = []  # [] => todas las disponibles. Ej: ['USDJPY']

# Candles
TIMESTAMP_COL = 'timestamp'
OPEN_COL = 'open'
HIGH_COL = 'high'
LOW_COL = 'low'
CLOSE_COL = 'close'
VOL_COL = 'volume'

# Segmentacion global
MIN_SEG_BARS = 6
MAX_SEG_BARS = 128

# Metodo 1: pivots -> impulsos
PIVOT_PROM_ATR_MULT = 0.55
MIN_IMPULSE_PIPS = 12

# Metodo 2: compresion + breakout
RANGE_WIN_MIN = 8
RANGE_WIN_MAX = 56
MAX_RANGE_PIPS = 40
BREAKOUT_LOOKAHEAD = 10
BREAKOUT_MIN_PIPS = 6

# Metodo 3: CUSUM shocks
CUSUM_K_SIGMA = 0.5
CUSUM_H_SIGMA = 3.8
CUSUM_SEG_HALF_WIN = 14

# Metodo 4: tendencias completas (multi-escala)
TREND_WIN_MIN = 16
TREND_WIN_MAX = 96
TREND_SCAN_STEP = 2
TREND_MIN_MOVE_PIPS = 20
TREND_MIN_R2 = 0.72

# Metodo 5: patrones de 3 techos/suelos
TRIPLE_PROM_ATR_MULT = 0.40
TRIPLE_TOL_PIPS = 12
TRIPLE_MIN_HEIGHT_PIPS = 18
TRIPLE_BREAK_LOOKAHEAD = 20
TRIPLE_BREAK_PIPS = 4

# Matrix Profile (opcional)
MATRIX_WINDOW = 48
MATRIX_TOP_DISCORDS = 20

# Ranking y seleccion final
TOP_SEGMENTS_PER_SYMBOL = 240
MAX_OVERLAP_RATIO = 0.55
MAX_PER_SUBTYPE = 55

# Embedding de secuencia
SEQ_LEN = 64

# Visualizacion
PLOT_SYMBOL = None  # None => usa primer simbolo disponible
PLOT_LAST_BARS = 1400
PLOT_TOP_SEGMENTS = 40

# Export
OUTPUT_DIR = Path('./phases/phase02_market_alphabet/output_segments_beyond_sota')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# =========================
# Helpers base
# =========================

def pip_size(symbol: str) -> float:
    s = symbol.upper()
    return 0.01 if s.endswith('JPY') else 0.0001


def robust_zscore(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    if mad < 1e-9:
        std = np.nanstd(x)
        if std < 1e-9:
            return np.zeros_like(x, dtype=float)
        z = (x - np.nanmean(x)) / (std + eps)
    else:
        z = (x - med) / (1.4826 * mad + eps)
    return np.clip(z, -6.0, 6.0)


def discover_assets(data_dir: Path, symbols: list[str]) -> list[tuple[str, Path]]:
    files = sorted(data_dir.glob('*.parquet'))
    rows = []
    allow = {s.upper() for s in symbols} if symbols else None
    for p in files:
        sym = p.stem.upper()
        if allow is not None and sym not in allow:
            continue
        rows.append((sym, p))
    return rows


def load_asset(path: Path) -> pd.DataFrame:
    df = pd.read_parquet(path).copy()
    required = {TIMESTAMP_COL, OPEN_COL, HIGH_COL, LOW_COL, CLOSE_COL}
    miss = [c for c in required if c not in df.columns]
    if miss:
        raise ValueError(f'Faltan columnas {miss} en {path}')

    df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL], errors='coerce')
    for c in [OPEN_COL, HIGH_COL, LOW_COL, CLOSE_COL]:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    if VOL_COL in df.columns:
        df[VOL_COL] = pd.to_numeric(df[VOL_COL], errors='coerce').fillna(0.0)

    df = df.dropna(subset=[TIMESTAMP_COL, CLOSE_COL])
    df = df.sort_values(TIMESTAMP_COL).drop_duplicates(subset=[TIMESTAMP_COL], keep='last').reset_index(drop=True)

    # Returns + ATR proxy
    df['ret'] = df[CLOSE_COL].pct_change().fillna(0.0)
    tr1 = (df[HIGH_COL] - df[LOW_COL]).abs()
    tr2 = (df[HIGH_COL] - df[CLOSE_COL].shift(1)).abs()
    tr3 = (df[LOW_COL] - df[CLOSE_COL].shift(1)).abs()
    df['tr'] = np.nanmax(np.vstack([tr1.values, tr2.values, tr3.values]), axis=0)
    df['atr14'] = pd.Series(df['tr']).rolling(14, min_periods=1).mean().values
    df['rv14'] = pd.Series(df['ret']).rolling(14, min_periods=5).std().fillna(0.0).values
    return df


def seg_basic_stats(df: pd.DataFrame, s: int, e: int, pip: float, method: str, subtype: str, base_score: float) -> dict:
    sl = df.iloc[s:e+1]
    c0 = float(sl[CLOSE_COL].iloc[0])
    c1 = float(sl[CLOSE_COL].iloc[-1])
    hi = float(sl[HIGH_COL].max())
    lo = float(sl[LOW_COL].min())
    n = int(e - s + 1)

    move_pips = (c1 - c0) / pip
    range_pips = (hi - lo) / pip

    rv_inside = float(sl['ret'].std(ddof=0)) if n > 1 else 0.0
    rv_prev = float(df['ret'].iloc[max(0, s-24):s].std(ddof=0)) if s > 2 else rv_inside
    vol_ratio = rv_inside / (rv_prev + 1e-12)

    x = np.arange(n, dtype=float)
    y = sl[CLOSE_COL].to_numpy(dtype=float)
    if n >= 3 and np.std(y) > 0:
        coef = np.polyfit(x, y, 1)
        yhat = coef[0] * x + coef[1]
        ss_res = np.sum((y - yhat) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2) + 1e-12
        trend_r2 = float(1.0 - ss_res / ss_tot)
        slope = float(coef[0])
    else:
        trend_r2 = 0.0
        slope = 0.0

    return {
        'method': method,
        'subtype': subtype,
        'start_idx': int(s),
        'end_idx': int(e),
        'start_ts': sl[TIMESTAMP_COL].iloc[0],
        'end_ts': sl[TIMESTAMP_COL].iloc[-1],
        'n_bars': n,
        'move_pips': float(move_pips),
        'abs_move_pips': float(abs(move_pips)),
        'range_pips': float(range_pips),
        'rv_inside': float(rv_inside),
        'rv_prev': float(rv_prev),
        'vol_ratio': float(vol_ratio),
        'trend_r2': float(trend_r2),
        'slope': float(slope),
        'score_base': float(base_score),
    }


def segment_to_sequence(close: np.ndarray, s: int, e: int, out_len: int = 64) -> np.ndarray:
    y = np.asarray(close[s:e+1], dtype=float)
    if len(y) < 2:
        y = np.array([y[0], y[0]], dtype=float) if len(y) else np.zeros(2, dtype=float)
    x_old = np.linspace(0.0, 1.0, num=len(y))
    x_new = np.linspace(0.0, 1.0, num=out_len)
    z = np.interp(x_new, x_old, y)
    z = (z - z.mean()) / (z.std() + 1e-12)
    return z


In [ ]:
# =========================
# Candidate generators
# =========================

def _pivot_points(close: np.ndarray, prominence_price: float):
    peaks, _ = find_peaks(close, prominence=prominence_price)
    troughs, _ = find_peaks(-close, prominence=prominence_price)
    return np.asarray(peaks, dtype=int), np.asarray(troughs, dtype=int)


def generate_pivot_impulses(df: pd.DataFrame, symbol: str) -> list[dict]:
    pip = pip_size(symbol)
    close = df[CLOSE_COL].to_numpy(dtype=float)
    atr = df['atr14'].to_numpy(dtype=float)

    prom_price = float(np.nanmedian(atr) * PIVOT_PROM_ATR_MULT)
    prom_price = max(prom_price, pip * 2.0)

    peaks, troughs = _pivot_points(close, prom_price)
    piv = np.unique(np.concatenate(([0], peaks, troughs, [len(close)-1])))
    piv.sort()

    out = []
    for a, b in zip(piv[:-1], piv[1:]):
        if b <= a:
            continue
        n = b - a + 1
        if n < MIN_SEG_BARS or n > MAX_SEG_BARS:
            continue
        move_pips = (close[b] - close[a]) / pip
        if abs(move_pips) < MIN_IMPULSE_PIPS:
            continue
        base = abs(move_pips) / math.sqrt(n)
        subtype = 'impulse_up' if move_pips > 0 else 'impulse_down'
        out.append(seg_basic_stats(df, a, b, pip, 'pivot', subtype, base))
    return out


def generate_range_breakouts(df: pd.DataFrame, symbol: str) -> list[dict]:
    pip = pip_size(symbol)
    close = df[CLOSE_COL].to_numpy(dtype=float)
    out = []
    n = len(df)

    for s in range(0, n - RANGE_WIN_MIN - BREAKOUT_LOOKAHEAD):
        best = None
        max_w = min(RANGE_WIN_MAX, n - s - BREAKOUT_LOOKAHEAD)
        for w in range(RANGE_WIN_MIN, max_w + 1):
            e = s + w - 1
            span = (close[s:e+1].max() - close[s:e+1].min()) / pip
            if span <= MAX_RANGE_PIPS:
                best = (e, span)
            else:
                break

        if best is None:
            continue

        e, span = best
        post_end = min(n - 1, e + BREAKOUT_LOOKAHEAD)
        post = close[e+1:post_end+1]
        if len(post) == 0:
            continue

        up_break = (post.max() - close[e]) / pip
        down_break = (close[e] - post.min()) / pip
        brk = max(up_break, down_break)
        if brk < BREAKOUT_MIN_PIPS:
            continue

        direction = 'up' if up_break >= down_break else 'down'
        seg_end = int(np.argmax(post) + e + 1) if direction == 'up' else int(np.argmin(post) + e + 1)
        if seg_end <= s:
            continue
        nseg = seg_end - s + 1
        if nseg < MIN_SEG_BARS or nseg > MAX_SEG_BARS:
            continue

        base = brk / (span + 1e-9)
        subtype = f'range_breakout_{direction}'
        out.append(seg_basic_stats(df, s, seg_end, pip, 'range_breakout', subtype, base))
    return out


def generate_cusum_shocks(df: pd.DataFrame, symbol: str) -> list[dict]:
    pip = pip_size(symbol)
    r = df['ret'].to_numpy(dtype=float)
    sig = np.nanstd(r) + 1e-12

    k = CUSUM_K_SIGMA * sig
    h = CUSUM_H_SIGMA * sig

    sp = 0.0
    sn = 0.0
    events = []
    for i, x in enumerate(r):
        sp = max(0.0, sp + x - k)
        sn = min(0.0, sn + x + k)
        if sp > h:
            events.append((i, 'shock_up', sp / sig))
            sp = 0.0
            sn = 0.0
        elif abs(sn) > h:
            events.append((i, 'shock_down', abs(sn) / sig))
            sp = 0.0
            sn = 0.0

    out = []
    n = len(df)
    for i, subtype, strength in events:
        s = max(0, i - CUSUM_SEG_HALF_WIN)
        e = min(n - 1, i + CUSUM_SEG_HALF_WIN)
        if e - s + 1 < MIN_SEG_BARS:
            continue
        out.append(seg_basic_stats(df, s, e, pip, 'cusum', subtype, strength))
    return out


def generate_trend_runs(df: pd.DataFrame, symbol: str) -> list[dict]:
    pip = pip_size(symbol)
    close = df[CLOSE_COL].to_numpy(dtype=float)
    n = len(df)
    out = []

    for s in range(0, max(1, n - TREND_WIN_MIN), TREND_SCAN_STEP):
        max_w = min(TREND_WIN_MAX, n - s)
        for w in range(TREND_WIN_MIN, max_w + 1, TREND_SCAN_STEP):
            e = s + w - 1
            if e >= n:
                break
            y = close[s:e+1]
            x = np.arange(len(y), dtype=float)
            coef = np.polyfit(x, y, 1)
            yhat = coef[0] * x + coef[1]
            ss_res = np.sum((y - yhat) ** 2)
            ss_tot = np.sum((y - y.mean()) ** 2) + 1e-12
            r2 = 1.0 - ss_res / ss_tot
            move_pips = (y[-1] - y[0]) / pip

            if r2 < TREND_MIN_R2:
                continue
            if abs(move_pips) < TREND_MIN_MOVE_PIPS:
                continue

            base = abs(move_pips) * (r2 ** 1.4) / math.sqrt(w)
            subtype = 'trend_up' if move_pips > 0 else 'trend_down'
            out.append(seg_basic_stats(df, s, e, pip, 'trend', subtype, base))

    return out


def generate_triple_patterns(df: pd.DataFrame, symbol: str) -> list[dict]:
    pip = pip_size(symbol)
    close = df[CLOSE_COL].to_numpy(dtype=float)
    high = df[HIGH_COL].to_numpy(dtype=float)
    low = df[LOW_COL].to_numpy(dtype=float)
    atr = df['atr14'].to_numpy(dtype=float)
    prom_price = max(float(np.nanmedian(atr) * TRIPLE_PROM_ATR_MULT), pip * 2.0)

    peaks, troughs = _pivot_points(close, prom_price)
    out = []

    # Triple top
    for i in range(len(peaks) - 2):
        p1, p2, p3 = int(peaks[i]), int(peaks[i+1]), int(peaks[i+2])
        if not (p1 < p2 < p3):
            continue
        if p3 - p1 < 10 or p3 - p1 > MAX_SEG_BARS:
            continue

        tops = np.array([close[p1], close[p2], close[p3]], dtype=float)
        if (tops.max() - tops.min()) / pip > TRIPLE_TOL_PIPS:
            continue

        v1 = low[p1:p2+1].min()
        v2 = low[p2:p3+1].min()
        neckline = max(v1, v2)
        height_pips = (tops.mean() - neckline) / pip
        if height_pips < TRIPLE_MIN_HEIGHT_PIPS:
            continue

        le = min(len(close)-1, p3 + TRIPLE_BREAK_LOOKAHEAD)
        br_rel = np.where(close[p3:le+1] < neckline - TRIPLE_BREAK_PIPS * pip)[0]
        end_idx = int(p3 + br_rel[0]) if len(br_rel) else p3
        base = float(height_pips + (6.0 if len(br_rel) else 0.0))
        out.append(seg_basic_stats(df, p1, end_idx, pip, 'pattern', 'triple_top', base))

    # Triple bottom
    for i in range(len(troughs) - 2):
        t1, t2, t3 = int(troughs[i]), int(troughs[i+1]), int(troughs[i+2])
        if not (t1 < t2 < t3):
            continue
        if t3 - t1 < 10 or t3 - t1 > MAX_SEG_BARS:
            continue

        bottoms = np.array([close[t1], close[t2], close[t3]], dtype=float)
        if (bottoms.max() - bottoms.min()) / pip > TRIPLE_TOL_PIPS:
            continue

        h1 = high[t1:t2+1].max()
        h2 = high[t2:t3+1].max()
        neckline = min(h1, h2)
        height_pips = (neckline - bottoms.mean()) / pip
        if height_pips < TRIPLE_MIN_HEIGHT_PIPS:
            continue

        le = min(len(close)-1, t3 + TRIPLE_BREAK_LOOKAHEAD)
        br_rel = np.where(close[t3:le+1] > neckline + TRIPLE_BREAK_PIPS * pip)[0]
        end_idx = int(t3 + br_rel[0]) if len(br_rel) else t3
        base = float(height_pips + (6.0 if len(br_rel) else 0.0))
        out.append(seg_basic_stats(df, t1, end_idx, pip, 'pattern', 'triple_bottom', base))

    return out


def generate_matrix_discords(df: pd.DataFrame, symbol: str) -> list[dict]:
    if not HAS_STUMPY:
        return []
    pip = pip_size(symbol)
    close = df[CLOSE_COL].to_numpy(dtype=np.float64)
    if len(close) <= MATRIX_WINDOW + 2:
        return []
    mp = stumpy.stump(close, m=MATRIX_WINDOW)
    profile = mp[:, 0].astype(float)
    idx = np.argsort(profile)[-MATRIX_TOP_DISCORDS:][::-1]

    out = []
    for i in idx:
        s = int(i)
        e = int(i + MATRIX_WINDOW - 1)
        if e - s + 1 < MIN_SEG_BARS or e >= len(df):
            continue
        out.append(seg_basic_stats(df, s, e, pip, 'matrix_profile', 'discord', float(profile[i])))
    return out


In [ ]:
# =========================
# Ranking + NMS + pseudo labels
# =========================

def overlap_ratio(a0: int, a1: int, b0: int, b1: int) -> float:
    ov = max(0, min(a1, b1) - max(a0, b0) + 1)
    if ov <= 0:
        return 0.0
    la = a1 - a0 + 1
    lb = b1 - b0 + 1
    return ov / float(min(la, lb))


def assign_proxy_label(row: pd.Series) -> str:
    st = row['subtype']
    if st in ('impulse_up', 'impulse_down', 'trend_up', 'trend_down', 'triple_top', 'triple_bottom'):
        return st
    if st.startswith('range_breakout_'):
        return st
    if st in ('shock_up', 'shock_down'):
        return st
    if row['range_pips'] <= MAX_RANGE_PIPS and abs(row['move_pips']) <= max(5.0, row['range_pips'] * 0.35):
        return 'range'
    return 'other'


def duration_bucket(n_bars: int) -> str:
    if n_bars <= 14:
        return 'short'
    if n_bars <= 42:
        return 'mid'
    return 'long'


def select_diverse_segments(df: pd.DataFrame, top_n: int, max_overlap_ratio: float, max_per_subtype: int) -> pd.DataFrame:
    keep_rows = []

    for sym, g in df.groupby('symbol', sort=False):
        g = g.sort_values('score', ascending=False).reset_index(drop=True)

        subtype_count = {}
        bucket_count = {'short': 0, 'mid': 0, 'long': 0}
        bucket_target = {
            'short': int(top_n * 0.30),
            'mid': int(top_n * 0.45),
            'long': int(top_n * 0.25),
        }

        kept = []

        # pass 1: enforce diversity
        for row in g.itertuples(index=False):
            if len(kept) >= top_n:
                break
            s0, e0 = int(row.start_idx), int(row.end_idx)
            b = duration_bucket(int(row.n_bars))
            st = row.subtype

            if subtype_count.get(st, 0) >= max_per_subtype:
                continue
            if bucket_count[b] >= bucket_target[b]:
                continue

            ok = True
            for k in kept:
                if overlap_ratio(s0, e0, k['start_idx'], k['end_idx']) > max_overlap_ratio:
                    ok = False
                    break
            if not ok:
                continue

            d = row._asdict()
            kept.append({'start_idx': s0, 'end_idx': e0})
            keep_rows.append(d)
            subtype_count[st] = subtype_count.get(st, 0) + 1
            bucket_count[b] += 1

        # pass 2: fill remaining without strict bucket quotas
        if len(kept) < top_n:
            for row in g.itertuples(index=False):
                if len(kept) >= top_n:
                    break
                s0, e0 = int(row.start_idx), int(row.end_idx)
                st = row.subtype

                if subtype_count.get(st, 0) >= max_per_subtype:
                    continue

                ok = True
                for k in kept:
                    if overlap_ratio(s0, e0, k['start_idx'], k['end_idx']) > max_overlap_ratio:
                        ok = False
                        break
                if not ok:
                    continue

                d = row._asdict()
                kept.append({'start_idx': s0, 'end_idx': e0})
                keep_rows.append(d)
                subtype_count[st] = subtype_count.get(st, 0) + 1

    return pd.DataFrame(keep_rows)


def kmeans_numpy(X: np.ndarray, k: int, n_iter: int = 40, seed: int = 42) -> tuple[np.ndarray, np.ndarray]:
    rs = np.random.RandomState(seed)
    n, d = X.shape
    if n == 0:
        return np.array([], dtype=int), np.empty((0, d), dtype=float)
    k = int(max(1, min(k, n)))

    init_idx = rs.choice(n, size=k, replace=False)
    centers = X[init_idx].copy()

    for _ in range(n_iter):
        dist = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        labels = dist.argmin(axis=1)
        new_centers = centers.copy()
        for j in range(k):
            mask = labels == j
            if np.any(mask):
                new_centers[j] = X[mask].mean(axis=0)
            else:
                new_centers[j] = X[rs.randint(0, n)]
        if np.allclose(new_centers, centers, atol=1e-6):
            centers = new_centers
            break
        centers = new_centers

    dist = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    labels = dist.argmin(axis=1)
    return labels, centers


In [ ]:
# =========================
# Run full pipeline
# =========================
assets = discover_assets(DATA_DIR, SYMBOLS)
if not assets:
    raise SystemExit(f'No se encontraron assets en {DATA_DIR} con SYMBOLS={SYMBOLS}')

all_candidates = []
close_cache = {}

for symbol, path in assets:
    df = load_asset(path)
    close_cache[symbol] = df[[TIMESTAMP_COL, CLOSE_COL]].copy()

    c1 = generate_pivot_impulses(df, symbol)
    c2 = generate_range_breakouts(df, symbol)
    c3 = generate_cusum_shocks(df, symbol)
    c4 = generate_matrix_discords(df, symbol)
    c5 = generate_trend_runs(df, symbol)
    c6 = generate_triple_patterns(df, symbol)

    candidates = c1 + c2 + c3 + c4 + c5 + c6
    for c in candidates:
        c['symbol'] = symbol
        c['asset_path'] = str(path)
    all_candidates.extend(candidates)

    print(f"{symbol}: piv={len(c1)} rb={len(c2)} cusum={len(c3)} matrix={len(c4)} trend={len(c5)} triple={len(c6)} total={len(candidates)}")

cand = pd.DataFrame(all_candidates)
if cand.empty:
    raise SystemExit('No se generaron segmentos. Ajusta umbrales.')

# score compuesto orientado a patrones completos (no solo velocidad)
a = robust_zscore(cand['abs_move_pips'].to_numpy(float))
r = robust_zscore(cand['range_pips'].to_numpy(float))
v = robust_zscore(cand['vol_ratio'].to_numpy(float))
t = robust_zscore(cand['trend_r2'].to_numpy(float))
b = robust_zscore(cand['score_base'].to_numpy(float))

n_bars = cand['n_bars'].to_numpy(float)
dur_pref = np.exp(-((n_bars - 30.0) / 22.0) ** 2)  # favorece segmentos medianos/completos

type_bonus = cand['subtype'].map({
    'triple_top': 0.6,
    'triple_bottom': 0.6,
    'trend_up': 0.5,
    'trend_down': 0.5,
    'impulse_up': 0.3,
    'impulse_down': 0.3,
    'range_breakout_up': 0.2,
    'range_breakout_down': 0.2,
    'shock_up': 0.2,
    'shock_down': 0.2,
}).fillna(0.0).to_numpy(float)

cand['score'] = (
    0.20 * a +
    0.14 * r +
    0.16 * v +
    0.16 * t +
    0.12 * b +
    0.14 * robust_zscore(dur_pref) +
    0.08 * type_bonus
)

cand = cand.replace([np.inf, -np.inf], np.nan).fillna(0.0)

# preseleccion balanceada por subtipo y luego seleccion diversa
cand = (
    cand.sort_values(['symbol', 'subtype', 'score'], ascending=[True, True, False])
        .groupby(['symbol', 'subtype'], as_index=False, group_keys=False)
        .head(MAX_PER_SUBTYPE * 2)
        .reset_index(drop=True)
)

cand = (
    cand.sort_values(['symbol', 'score'], ascending=[True, False])
        .groupby('symbol', as_index=False, group_keys=False)
        .head(TOP_SEGMENTS_PER_SYMBOL * 5)
        .reset_index(drop=True)
)

selected = select_diverse_segments(
    cand,
    top_n=TOP_SEGMENTS_PER_SYMBOL,
    max_overlap_ratio=MAX_OVERLAP_RATIO,
    max_per_subtype=MAX_PER_SUBTYPE,
)

selected = selected.sort_values(['symbol', 'score'], ascending=[True, False]).reset_index(drop=True)
selected['proxy_label'] = selected.apply(assign_proxy_label, axis=1)

# secuencias para clasificacion
seqs = []
for row in selected.itertuples(index=False):
    sym = row.symbol
    s = int(row.start_idx)
    e = int(row.end_idx)
    c = close_cache[sym][CLOSE_COL].to_numpy(dtype=float)
    seqs.append(segment_to_sequence(c, s, e, out_len=SEQ_LEN))

X_seq = np.vstack(seqs) if seqs else np.empty((0, SEQ_LEN), dtype=float)
selected['seq_id'] = np.arange(len(selected), dtype=int)

# clustering rapido sobre embedding (sin sklearn)
if len(X_seq) >= 8:
    k = int(min(9, max(4, round(math.sqrt(len(X_seq))))))
    labels, centers = kmeans_numpy(X_seq, k=k, n_iter=60, seed=42)
    selected['cluster_id'] = labels.astype(int)
else:
    selected['cluster_id'] = -1

print('\nResumen final:')
print(selected.groupby('symbol').size().rename('segments').to_string())
print('\nProxy labels:')
print(selected['proxy_label'].value_counts().to_string())
print('\nSubtypes top:')
print(selected['subtype'].value_counts().head(20).to_string())

print('\nPreview:')
display(selected.head(25))


In [ ]:
# =========================
# Visualizacion de segmentos top
# =========================
if selected.empty:
    raise SystemExit('No hay segmentos seleccionados')

sym = PLOT_SYMBOL or selected['symbol'].iloc[0]
plot_df = load_asset(next(p for s, p in assets if s == sym))
sub = selected[selected['symbol'] == sym].head(PLOT_TOP_SEGMENTS).copy()

if plot_df.empty:
    raise SystemExit(f'Sin data para {sym}')

plot_tail = plot_df.iloc[-min(PLOT_LAST_BARS, len(plot_df)):].copy()
start_idx_min = int(plot_tail.index.min())

color_map = {
    'triple_top': 'darkred',
    'triple_bottom': 'darkgreen',
    'trend_up': 'forestgreen',
    'trend_down': 'firebrick',
    'impulse_up': 'limegreen',
    'impulse_down': 'tomato',
    'range_breakout_up': 'teal',
    'range_breakout_down': 'orange',
    'shock_up': 'navy',
    'shock_down': 'purple',
}

plt.figure(figsize=(18, 7))
plt.plot(plot_tail[TIMESTAMP_COL], plot_tail[CLOSE_COL], color='black', lw=1.0, label=f'{sym} close')

for _, r in sub.iterrows():
    s = int(r['start_idx'])
    e = int(r['end_idx'])
    if e < start_idx_min:
        continue
    st = r['subtype']
    color = color_map.get(st, 'gray')
    alpha = float(min(0.32, max(0.08, 0.10 + 0.03 * r['score'])))
    plt.axvspan(plot_df.loc[s, TIMESTAMP_COL], plot_df.loc[e, TIMESTAMP_COL], color=color, alpha=alpha)

plt.title(f'Segmentos importantes y completos - {sym} (top={len(sub)})')
plt.xlabel('timestamp')
plt.ylabel('price')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print('Subtypes mostrados:')
print(sub['subtype'].value_counts().to_string())


In [ ]:
# =========================
# Export para clasificacion
# =========================
catalog = selected.copy()

# Guardar metadata/features (sin secuencias crudas)
meta_path = OUTPUT_DIR / 'segments_catalog.parquet'
catalog.to_parquet(meta_path, index=False)

# Guardar arrays para modelos
npz_path = OUTPUT_DIR / 'segments_ml_bundle.npz'
np.savez_compressed(
    npz_path,
    X_seq=X_seq.astype(np.float32),
    seq_id=catalog['seq_id'].to_numpy(dtype=np.int32),
    cluster_id=catalog['cluster_id'].to_numpy(dtype=np.int32),
)

# Un mini resumen por label/cluster
summary = (
    catalog.groupby(['proxy_label', 'cluster_id'], as_index=False)
           .agg(n_segments=('seq_id', 'count'),
                mean_score=('score', 'mean'),
                mean_abs_move=('abs_move_pips', 'mean'),
                mean_range=('range_pips', 'mean'))
           .sort_values('n_segments', ascending=False)
)
summary_path = OUTPUT_DIR / 'segments_summary.parquet'
summary.to_parquet(summary_path, index=False)

print('Export listo:')
print('-', meta_path)
print('-', npz_path)
print('-', summary_path)

display(summary.head(20))
